In [3]:
import jax.numpy as jnp
import roughpy_jax as rpj
import numpy as np
import jax
from roughpy_jax.streams import LieIncrementStream
from roughpy_jax.streams.lie_increment_stream import _zero_lie
from roughpy_jax.intervals import IntervalType, Partition
from roughpy_jax.streams.piecewise_abelian_stream import to_piecewise_abelian_stream
from roughpy_jax.algebra import to_signature, antipode, to_log_signature, lie_to_tensor, as_free_tensor, _remove_unit_term
from roughpy_jax.dense_algebra import get_batch_shape, _algebra_scalar_multiply, broadcast_to_batch_shape
from utils import generate_linear_batch, generate_trig_batch, generate_random_batch

In [4]:
times = [[0., 0.5, 1.0], [0.,0.5, 1.0], [0.,0.5, 1.0], [0.,0.5, 1.0]]
data = [jnp.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], dtype=jnp.float32), 
        jnp.array([[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]], dtype=jnp.float32), 
        jnp.array([[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]], dtype=jnp.float32),
        jnp.array([[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]], dtype=jnp.float32)]
Lie_Basis = rpj.LieBasis(width=2, depth=2)
Tensor_Basis = rpj.to_tensor_basis(Lie_Basis)
X_Lie = LieIncrementStream.from_increments(timestamps=times, data=data, input_data_basis=None, resolution=2, lie_basis=Lie_Basis)

In [10]:
len(data[0][0])

2

In [16]:
# creates an array of each pair we need to calculate for the Gram matrix
B = 4
pairs = jnp.stack(jnp.triu_indices(B), axis=1)

In [18]:
L=3
n=2
intervals = uniform_intervals(L)
X_LSPs = tuple(lie_to_tensor(X_Lie.log_signature(interval)) for interval in intervals)
X_SPs = tuple(rpj.ft_exp(x_ls, out_basis=x_ls.basis) for x_ls in X_LSPs)
X_LSPTs = trunc(X_LSPs, n, n-1)
X_SPTs = trunc(X_SPs, n, n-1)
X_SPTs_zero = tuple(_remove_unit_term(x) for x in X_SPTs) 

In [19]:
def initialise_PDE(X_SPTs_zero, Y_SPTs_zero, Tensor_Basis):
    # we take in X_SPs and Y_SPs as shape (M, T) where M = B(B+1)/2
    L = len(X_SPTs_zero)
    M, _ = X_SPTs_zero[0].shape

    K = jnp.zeros((L+1, L+1, M), dtype=jnp.float32) 
        # Since the paper gives K[0, v] as the inner product of Z_0^x and Z_v^y and analogous for K[u, 0], I assume we are setting Z_0^x = 1 = Z_0^y where 1 = (1,0,0,0,...) 
        # is in the signature sense
    
    K = K.at[0, :, :].set(1)
    K = K.at[:, 0, :].set(1)
    zero_tensor = rpj.FreeTensor.zero(basis=Tensor_Basis, batch_dims=(M,))
    phi = [[zero_tensor]*(L+1)]*(L+1)  
    psi = [[zero_tensor]*(L+1)]*(L+1)

    for i in range(1, L+1):
        phi[i][0] = X_SPTs_zero[i-1]
    for j in range(1, L+1):
        psi[0][j] = Y_SPTs_zero[j-1]
    return phi, psi, K

In [20]:
# helper function for taking pairs and then turning into tensor

def ft_pairs(tuple_of_arrays, pairs, order, tensor_basis):
    return tuple(rpj.FreeTensor(jnp.asarray(array)[pairs[:, order]], tensor_basis) for array in tuple_of_arrays) 

In [21]:
X_SPTs_zero = ft_pairs(X_SPTs_zero, pairs, 0, Tensor_Basis)
Y_SPTs_zero = ft_pairs(X_SPTs_zero, pairs, 1, Tensor_Basis)

In [22]:
phi, psi, K = initialise_PDE(X_SPTs_zero, Y_SPTs_zero, Tensor_Basis)

In [ ]:
# Helper functions used to help calculations present in algorithm 5.1

def inner_prod(X,Y): 
    return jnp.sum(X*Y)  

def batched_inner(X, Y):
    return jax.jit(jax.vmap(inner_prod, in_axes=(0, 0)))(X.__array__(), Y.__array__())

def right_adj(A,C, tensor_basis):

    ant_A = antipode(A) # check whether antipode applies element wise in a batch
    ant_C = antipode(C)
    
    left_adj = rpj.ft_adjoint_left_mul(ant_A, ant_C)

    left_adj = rpj.FreeTensor(left_adj, basis=tensor_basis)  
    
    adj_A_C = antipode(left_adj)
    
    return adj_A_C

    
def eval_adj(phi, psi, x, y, tensor_basis):

    r_x_y = right_adj(x, y, tensor_basis)  
    r_y_x = right_adj(y, x, tensor_basis)

    return batched_inner(phi, r_x_y) + batched_inner(psi, r_y_x)

def add_tensor_scalar(a, s): 
# ---------------------------------------------------------------------------------------------
# --------------------------------- Potential weak point!! ------------------------------------
#----------------------------------------------------------------------------------------------
    cls = type(a)
    scalar = jnp.asarray(s)
    ext_scalar = broadcast_to_batch_shape(scalar, a.batch_shape)
    result_data = jnp.add(a.data, ext_scalar)
    return cls(result_data, a.basis)

Take the various X inputs as rpj.FreeTensor(jnp.asarray(X_LSPTs[i])[pairs[:, 0]], Tensor_Basis) and the various Y inputs as rpj.FreeTensor(jnp.asarray(X_LSPTs[i])[pairs[:, 1]], Tensor_Basis). Can do the same with phi, psi and K (this lets us avoid vmap all together)

In [ ]:
def compute_phi(xi, xti, phi01, psi01, K00):
    phi11 = phi01 + xti.__mul__(K00)\
            + rpj.ft_mul(phi01, xti)\
            + add_tensor_scalar(as_free_tensor(rpj.ft_adjoint_left_mul(psi01, xi)), -batched_inner(psi01, xti))

    return phi11

def compute_psi(yj, ytj, phi10, psi10, K00):
    psi11 = psi10 + ytj.__mul__(K00)\
            + rpj.ft_mul(psi10, ytj)\
            + add_tensor_scalar(as_free_tensor(rpj.ft_adjoint_left_mul(phi10, yj)), - batched_inner(phi10, ytj))
    return psi11

def compute_K(xi, yj, phi00, phi01, phi10, phi11, psi00, psi01, psi10, psi11, K00, K01, K10, tensor_basis):

    eval_adj_ = eval_adj(phi00, psi00, xi, yj, tensor_basis)
    next_eval_adj = eval_adj(phi11, psi11, xi, yj, tensor_basis) # maybe these can be done with built in rpj methods
    temp_2 =  eval_adj(phi01, psi01, xi, yj, tensor_basis)
    temp_3 = eval_adj(phi10, psi10, xi, yj, tensor_basis)

    G = batched_inner(xi, yj)
    f_1 = K00 * G + eval_adj_
    f_2 = K01 * G + temp_2 
    f_3 = K10 * G + temp_3 

    u_p = K10 + K01 - K00 + f_1
    f_p = u_p * G + next_eval_adj

    K11 = K10 + K01 - K00 + (1./4)*(f_1 + f_2 + f_3 + f_p)
    return K11

In [25]:
xlspts = ft_pairs(X_LSPTs, pairs, 0, Tensor_Basis)
ylspts = ft_pairs(X_LSPTs, pairs, 1, Tensor_Basis)
xlsps = ft_pairs(X_LSPs, pairs, 0, Tensor_Basis)
ylsps = ft_pairs(X_LSPs, pairs, 1, Tensor_Basis)

In [28]:
for i in range(L):
    for j in range(L):
        xi = xlsps[i]
        yj = ylsps[j]
        xti = xlspts[i]
        ytj = ylspts[j]
        phi00, phi01, phi10 = phi[i][j], phi[i][j+1], phi[i+1][j]
        psi00, psi01, psi10 = psi[i][j], psi[i][j+1], psi[i+1][j]
        K00, K01, K10 = K[i][j], K[i][j+1], K[i+1][j]
        phi11 = compute_phi(xi, xti, phi01, psi01, K00)
        psi11 = compute_psi(yj, ytj, phi10, psi10, K00)
        K11 = compute_K(xi, yj, phi00, phi01, phi10, phi11, psi00, psi01, psi10, psi11, K00, K01, K10, Tensor_Basis)
        phi[i+1][j+1] = phi11
        psi[i+1][j+1] = psi11
        K = K.at[i+1, j+1].set(K11)     

In [29]:
K[-1, -1, :]

Array([  12.25,  156.25,  156.25,  156.25, 3306.25, 3306.25, 3306.25,
       3306.25, 3306.25, 3306.25], dtype=float32)

In [35]:
X = rpj.FreeTensor(jnp.asarray(lie_to_tensor(X_Lie.log_signature()))[1], Tensor_Basis)
Y = rpj.FreeTensor(jnp.asarray(lie_to_tensor(X_Lie.log_signature()))[1], Tensor_Basis)
k = rpj.tensor_pairing(X, Y)
(rpj.tensor_pairing(X_Lie.signature(), X_Lie.signature()), k)

(Array([ 12914.25, 665072.25, 665072.25, 665072.25], dtype=float32),
 Array(1661., dtype=float32))

In [ ]:
a = as_free_tensor(rpj.ft_adjoint_left_mul(psi01, xi))
s = batched_inner(psi01, xti)
cls = type(a)
scalar = jnp.asarray(s)
ext_scalar = broadcast_to_batch_shape(scalar, a.batch_shape)
result_data = jnp.add(a.data, ext_scalar)
cls(result_data, a.basis).__array__()
a.__array__()

In [ ]:
def make_one_dim(batch_tensor, dim, tensor_basis):
    return rpj.FreeTensor(jnp.asarray(batch_tensor)[dim], tensor_basis)

In [ ]:
# one-dimensional test case

for i in range(L):
    for j in range(L):
        xoi = make_one_dim(xlsps[i], 0, Tensor_Basis)
        yoj = make_one_dim(ylsps[j], 0, Tensor_Basis)
        xoti = make_one_dim(xlspts[i], 0, Tensor_Basis)
        yotj = make_one_dim(ylspts[j], 0, Tensor_Basis)
        phi00, phi01, phi10 = phi[i][j], phi[i][j+1], phi[i+1][j]
        psi00, psi01, psi10 = psi[i][j], psi[i][j+1], psi[i+1][j]
        K00, K01, K10 = K[i][j], K[i][j+1], K[i+1][j]
        phi11 = compute_phi(xi, xti, phi01, psi01, K00)
        psi11 = compute_psi(yj, ytj, phi10, psi10, K00)
        K11 = compute_K(xi, yj, phi00, phi01, phi10, phi11, psi00, psi01, psi10, psi11, K00, K01, K10, Tensor_Basis)
        phi[i+1][j+1] = phi11
        psi[i+1][j+1] = psi11
        K = K.at[i+1, j+1].set(K11)            

Array([-31116.5, -31116.5, -31116.5, -31116.5, -31116.5, -31116.5,
       -31116.5, -31116.5, -31116.5, -31116.5], dtype=float32)

Next steps:
Make stuff in previous cell into functions, as well as breaking down exactly what it does and checking correctness.
Broadcast the size M final outputs of K into into the B x B Gram matrix
Remove bloat from notebook
After this try jax.JIT, and jax.lax.fori_loop to see if they help